## RNN Exercises

### 1. Build a simple RNN from scratch with NumPy.

In [2]:
import numpy as np

class RNN:
    def __init__(self, d_in, d_h, d_out, lr=0.01):
        # Weights & biases
        self.Wxh = np.random.randn(d_h, d_in) * 0.1
        self.Whh = np.random.randn(d_h, d_h) * 0.1
        self.Why = np.random.randn(d_out, d_h) * 0.1
        self.bh, self.by = np.zeros((d_h, 1)), np.zeros((d_out, 1))
        self.lr = lr

    def forward(self, xs):
        # xs shape: (seq_len, d_in, 1)
        self.xs, self.hs, self.ys = xs, {}, {}
        self.hs[-1] = np.zeros((self.Whh.shape[0], 1))

        for t, x in enumerate(xs):
            self.hs[t] = np.tanh(self.Wxh @ x + self.Whh @ self.hs[t-1] + self.bh)
            self.ys[t] = self.Why @ self.hs[t] + self.by
        return self.ys

    def backward(self, targets):
        # target shape: (seq_len, d_out, 1)
        dWxh, dWhh, dWhy = np.zeros_like(self.Wxh), np.zeros_like(self.Whh), np.zeros_like(self.Why)
        dbh, dby = np.zeros_like(self.bh), np.zeros_like(self.by)
        dh_next = np.zeros_like(self.bh)

        for t in reversed(range(len(targets))):
            dy = self.ys[t] - targets[t]  # dLoss / dy (MSE derivative)
            dWhy += dy @ self.hs[t].T
            dby += dy

            dh = self.Why.T @ dy + dh_next  # Backprop to hidden state

            dh_raw = (1 - self.hs[t] ** 2) * dh

            dWxh += dh_raw @ self.xs[t].T
            dWhh += dh_raw @ self.hs[t-1].T
            dbh += dh_raw
            dh_next = self.Whh.T @ dh_raw  # Pass gradient to t-1

        # SGD step
        for param, dparam in zip([self.Wxh, self.Whh, self.Why, self.bh, self.by], [dWxh, dWhh, dWhy, dbh, dby]):
            param -= self.lr * np.clip(dparam, -5.0, 5.0)

# Pass 3 time-steps of scalar input
rnn = RNN(d_in=1, d_h=4, d_out=1)
xs = [np.array([[i]]) for i in [1.0, 2.0, 3.0]]
ys = [np.array([[i]]) for i in [2.0, 3.0, 4.0]]

rnn.forward(xs)
rnn.backward(ys)